<a href="https://colab.research.google.com/github/lawrancesahayasundar-tech/my_first_repo/blob/main/task1_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# task1_data_collection.py

"""
TrendPulse - Task 1: Data Collection

This script fetches trending stories from HackerNews API,
categorizes them based on keywords, and saves them into a JSON file.

Author: Lawrence
"""

import requests
import json
import os
import time
from datetime import datetime

# Base URLs for HackerNews API
TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEM_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"

# Custom header (as required)
HEADERS = {"User-Agent": "TrendPulse/1.0"}

# Category keyword mapping
CATEGORY_KEYWORDS = {
    "technology": ["ai", "software", "tech", "code", "computer", "data", "cloud", "api", "gpu", "llm"],
    "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "global"],
    "sports": ["nfl", "nba", "fifa", "sport", "game", "team", "player", "league", "championship"],
    "science": ["research", "study", "space", "physics", "biology", "discovery", "nasa", "genome"],
    "entertainment": ["movie", "film", "music", "netflix", "game", "book", "show", "award", "streaming"]
}

# Max stories per category
MAX_PER_CATEGORY = 25


def fetch_top_story_ids():
    """Fetch top story IDs from HackerNews"""
    try:
        response = requests.get(TOP_STORIES_URL, headers=HEADERS)
        response.raise_for_status()
        return response.json()[:500]  # first 500 IDs
    except Exception as e:
        print(f"Error fetching top stories: {e}")
        return []


def fetch_story(story_id):
    """Fetch individual story details"""
    try:
        url = ITEM_URL.format(story_id)
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Failed to fetch story {story_id}: {e}")
        return None


def categorize_title(title):
    """Assign category based on keywords"""
    if not title:
        return None

    title_lower = title.lower()

    for category, keywords in CATEGORY_KEYWORDS.items():
        for keyword in keywords:
            if keyword in title_lower:
                return category

    return None  # ignore if no match


def main():
    print("Fetching top stories...")

    story_ids = fetch_top_story_ids()
    collected_data = []

    # Track count per category
    category_counts = {cat: 0 for cat in CATEGORY_KEYWORDS}

    for category in CATEGORY_KEYWORDS:

        print(f"\nProcessing category: {category}")

        for story_id in story_ids:

            # Stop if category limit reached
            if category_counts[category] >= MAX_PER_CATEGORY:
                break

            story = fetch_story(story_id)

            if not story or "title" not in story:
                continue

            detected_category = categorize_title(story["title"])

            # Only collect if it matches current category
            if detected_category == category:

                record = {
                    "post_id": story.get("id"),
                    "title": story.get("title"),
                    "category": category,
                    "score": story.get("score", 0),
                    "num_comments": story.get("descendants", 0),
                    "author": story.get("by", "unknown"),
                    "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                }

                collected_data.append(record)
                category_counts[category] += 1

        # Required delay between categories
        time.sleep(2)

    # Create data folder if not exists
    if not os.path.exists("data"):
        os.makedirs("data")

    # File name with date
    filename = f"data/trends_{datetime.now().strftime('%Y%m%d')}.json"

    # Save JSON
    with open(filename, "w") as f:
        json.dump(collected_data, f, indent=4)

    print(f"\nCollected {len(collected_data)} stories.")
    print(f"Saved to {filename}")


if __name__ == "__main__":
    main()

Fetching top stories...

Processing category: technology

Processing category: worldnews

Processing category: sports

Processing category: science

Processing category: entertainment

Collected 86 stories.
Saved to data/trends_20260417.json
